In [1]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Literal

from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END

import sys
sys.path.append("../ingestion/python/data-enrichment-LLM")
from silver_enrichment import JobOfferState, Extract, CompanyOutput, LLM
sys.path.append("../ingestion/python/src")
from database import Database



c:\APPS\Python312\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
db = Database()
query = """
    SELECT 
        rjo.id_job, 
        rjo.api_source, 
        rjo.job_title, 
        rjo.contract_type, 
        rjo.job_publisher, 
        rjo.company, 
        rjo.company_website, 
        rjo.location_raw,
        rjo.city,
        rjo.country,
        rjo.latitude,
        rjo.longitude,
        rjo.is_remote,
        rjo.offer_description,
        rjo.offer_url,
        rjo.source_platform,
        rjo.published_at,
        rjo.collected_at
    FROM raw.job_offer rjo
    WHERE rjo.company = ''
    LIMIT 10;
"""

line = db.execute(query)

In [3]:
line

[{'id_job': 'cj_c54e2eeee7d8ae0c7ae109464f711008',
  'api_source': 'careerjet',
  'job_title': 'Environmental Remediation Project Coordinator (Santiago)',
  'contract_type': None,
  'job_publisher': None,
  'company': '',
  'company_website': None,
  'location_raw': 'Santiago, Región Metropolitana',
  'city': 'Santiago, Región Metropolitana',
  'country': 'CL',
  'latitude': None,
  'longitude': None,
  'is_remote': None,
  'offer_description': 'Description  Our Environmental business in Santiago is looking for an Environmental <b>Engineer</b>, Environmental Scientist... may include the collection and synthesizing of samples and environmental <b>data</b>, and recommend action based on <b>data</b> derived',
  'offer_url': 'https://jobviewtrack.com/v2/3EFs_b6aZoD9Cop_7XWz3adBP9We3KKxbsX7Bhu7OR4EK4jTDotILJkCJ3nKoSVIqfGIh3oNT-gs067ailjg8qU3Tux57cnFKhKWIQvZqPgOBRO1ByKEmdK1pTaGyzoZQL6j84REdgw4A0DyFlb7jXSnWew5pU9H-noLjVsecxGWkxnHuJ7QKHn1dEvUCE9KdeJhZid3mVbDv0dpETAmEdzqRdyhjSvY2sGG3oKKvEQ',
  

In [4]:
llm = LLM()
extract_company = Extract(
    llm=llm.llama4_smart,
    task='Find the name of the company who recruit for this job offer based on the title and the description of the job.',
    output_key='company_name',
    schema=CompanyOutput,
    fields=['job_title', 'offer_description']
)

builder = StateGraph(JobOfferState)
builder.add_node("extract_company", extract_company)
builder.add_edge(START, "extract_company")
builder.add_edge("extract_company", END)

graph = builder.compile()

In [5]:
for i in range(len(line)):
    real_state = line[i]
    result = graph.invoke(real_state)
    print(f"id={result['id_job']}, company_name={result['company_name']}")
    

id=cj_c54e2eeee7d8ae0c7ae109464f711008, company_name=null
id=cj_59cf47c22d3120c4c544a4cabfc915ca, company_name=Cencosud Media
id=cj_786203f7c225b3036202b80975d2206b, company_name=null
id=cj_fef20ce3a0164e2b99fa0c3c3fdd577a, company_name=Thoughtworks
id=cj_19971103e452c44dd99b69afe7026187, company_name=null
id=cj_fdd47bba11be1f542a3df663db7324ee, company_name=Experian
id=cj_48afdfe1187458d5f203eda5f2c2609f, company_name=Experian plc
id=cj_7600d1af4cb8b5b7595d6fe41bdc9bfd, company_name=Empresa confidencial
id=cj_53f06653ca5c39aedf775869b0ca821e, company_name=null
id=cj_1b243354ebeccfaff9f081c9517b9f5e, company_name=null


```python
llama3
id=cj_c54e2eeee7d8ae0c7ae109464f711008, company_name=null 
id=cj_59cf47c22d3120c4c544a4cabfc915ca, company_name=Cencosud 
id=cj_786203f7c225b3036202b80975d2206b, company_name=Microsoft 
id=cj_fef20ce3a0164e2b99fa0c3c3fdd577a, company_name=Thoughtworks 
id=cj_19971103e452c44dd99b69afe7026187, company_name=null 
id=cj_fdd47bba11be1f542a3df663db7324ee, company_name=Experian 
id=cj_48afdfe1187458d5f203eda5f2c2609f, company_name=Experian 
id=cj_7600d1af4cb8b5b7595d6fe41bdc9bfd, company_name=Empresa confidencial 
id=cj_53f06653ca5c39aedf775869b0ca821e, company_name=AECOM 
id=cj_1b243354ebeccfaff9f081c9517b9f5e, company_name=None

llama4 (GRAND VAINQUEUR)
id=cj_c54e2eeee7d8ae0c7ae109464f711008, company_name=null
id=cj_59cf47c22d3120c4c544a4cabfc915ca, company_name=Cencosud Media
id=cj_786203f7c225b3036202b80975d2206b, company_name=null
id=cj_fef20ce3a0164e2b99fa0c3c3fdd577a, company_name=Thoughtworks
id=cj_19971103e452c44dd99b69afe7026187, company_name=null
id=cj_fdd47bba11be1f542a3df663db7324ee, company_name=Experian
id=cj_48afdfe1187458d5f203eda5f2c2609f, company_name=Experian plc
id=cj_7600d1af4cb8b5b7595d6fe41bdc9bfd, company_name=Empresa confidencial
id=cj_53f06653ca5c39aedf775869b0ca821e, company_name=null
id=cj_1b243354ebeccfaff9f081c9517b9f5e, company_name=null
```

In [6]:
result

{'id_job': 'cj_1b243354ebeccfaff9f081c9517b9f5e',
 'api_source': 'careerjet',
 'job_title': 'Ingeniero Senior Back-end',
 'offer_description': 'behaviors--dark-mode-switch#select" <b>data</b>-behaviors--dark-mode-switch-target="option" <b>data</b>-preference="system" role...="menuitemradio" type="button"&gt; System  behaviors--dark-mode-switch#select" <b>data</b>-behaviors--dark-mode-switch-target="option" <b>data</b>',
 'contract_type': None,
 'is_remote': None,
 'job_publisher': None,
 'location_raw': 'Santiago, Región Metropolitana',
 'offer_url': 'https://jobviewtrack.com/v2/pZL0R2VaiXsUOurktAjyJJQ7f2pzcvkcMZDRJS9CCI5P5Li6kmswjX95I-Hbn6KvxyKLxm1n8gPiirUr7kqnQUrREwyV3SDQs3jPsTHv4BnYH3yWmDCNKC3UbJ9PG3NUsBzBLOHEz39r6uoqtajKCXgwiPpO4KEjFi6cOcmz_TEcLf8e_F5KxoKBsHrZa5BSkltbWziRbaj6O7tXOuaP97WBvwRM0Kfsr3BFDtAfRHY',
 'source_platform': None,
 'published_at': '2026-06-07 06:00:15+00',
 'collected_at': '2026-06-09 22:45:48.488771+00',
 'company_name': 'null',
 'company_website': None,
 'city